# Explanation of my code

## TEKO Vision + Attention + YawAux — OPTIMAL CONFIG (Final)

**Hyperparameters:** from Optuna *Trial 80* (reached **S41 / 180°**)  
**Logging:** TensorBoard enabled (for thesis figures)

---

### Author
Alexandre Schleier Neves da Silva


=======================================================================================

## 1. Imports & Environment Setup

### CUDA memory allocation (avoid fragmentation)
Configures CUDA allocator to reduce fragmentation:
- `expandable_segments:True` allows dynamic GPU memory growth
- `max_split_size_mb:128` limits block splitting

### Immediate log flushing
`print = partial(print, flush=True)` ensures prints appear immediately (useful for SLURM logs).

### Optional TensorBoard
Tries to load TensorBoard; if unavailable, disables logging without crashing.


In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

import argparse
import sys
import math
import socket
import time
import csv
from collections import deque
from functools import partial
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import random
from isaaclab.app import AppLauncher
print = partial(print, flush=True)

# TensorBoard
try:
    from torch.utils.tensorboard import SummaryWriter
    HAS_TENSORBOARD = True
except ImportError:
    HAS_TENSORBOARD = False
    print("[WARN] TensorBoard not available")


## 2. Hyperparameter Configuration

### Parameter table

| Parameter | Value | Purpose |
|---|---:|---|
| `learning_rate` | 0.00016 | Adam optimizer step size |
| `entropy_coef` | 0.0062 | Encourages exploration |
| `gae_lambda` | 0.94 | Bias–variance tradeoff in GAE |
| `gamma` | 0.99 | Discount factor for future rewards |
| `clip_ratio` | 0.2 | PPO clipping range (≈ [0.8, 1.2]) |
| `value_coef` | 0.5 | Value loss weight |
| `max_grad_norm` | 0.5 | Gradient clipping |
| `epochs` | 5 | PPO epochs per update |
| `batch_size` | 1024 | Minibatch size |
| `aux_yaw_coef` | 0.31 | Weight for auxiliary yaw prediction loss |
| `num_envs` | 120 | Parallel simulation environments |
| `rollout_len` | 128 | Steps collected before each PPO update |
| `advance_threshold` | 0.75 | Success rate needed to advance curriculum |
| `min_steps_before_advance` | 200,000 | Minimum training steps before stage advancement |
| `max_stage` | 41 | Final stage (corresponds to 180° yaw offset) |
| `log_interval` | 50,000 | Logging interval (TensorBoard/prints) |
| `save_interval` | 2,000,000 | Checkpoint saving interval |
| `max_steps` | 200,000,000 | Maximum total training steps |
| `max_hours` | 168 | Maximum wall-clock training time |

In [ ]:
CONFIG = {
    "max_steps": 200_000_000,
    "max_hours": 168,  # 7 days
    
    # OPTIMAL from Optuna Trial 80
    "learning_rate": 0.00016,
    "entropy_coef": 0.0062,
    "gae_lambda": 0.94,
    "gamma": 0.99,
    "clip_ratio": 0.2,
    "value_coef": 0.5,
    "max_grad_norm": 0.5,
    "epochs": 5,
    "batch_size": 1024,
    
    "aux_yaw_coef": 0.31,
    
    "num_envs": 120,
    "rollout_len": 128,
    
    "advance_threshold": 0.75,
    "min_steps_before_advance": 200_000,
    "max_stage": 41,
    
    "log_interval": 50_000,
    "save_interval": 2_000_000,
}

## 3. Spatial Attention Module

This module learns a **2D spatial attention mask** to highlight *where* the policy should focus in the image.

### What it does
Given a feature map `x` with shape **[B, C, H, W]**:
1. A **1×1 convolution** collapses channels and produces logits with shape **[B, 1, H, W]**
2. A **sigmoid** maps logits to attention weights in **[0, 1]**
3. The mask is applied via element-wise multiplication: **x ⊙ mask**

### Why it matters in TEKO docking
Docking is visually localized: the relevant cue is typically the **connector / docking interface region**.  
Spatial attention encourages the network to amplify features around that region and suppress irrelevant background.


In [ ]:
class SpatialAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 1, kernel_size=1)
    
    def forward(self, x):
        return x * torch.sigmoid(self.conv(x))


## 4. Channel Attention Module (Squeeze-and-Excitation)

This module learns **which feature channels** are most important (SE-style attention).  
While spatial attention answers **"where to look"**, channel attention answers **"what to look for"**.

### What it does
Given a feature map `x` with shape **[B, C, H, W]**:

1. **Squeeze (Global Average Pooling)**  
   Reduces spatial dimensions:
   - **[B, C, H, W] → [B, C]**

2. **Excitation (bottleneck MLP)**  
   A small MLP learns per-channel weights:
   - compress: `C → C / reduction`
   - expand: `C / reduction → C`

3. **Gate (sigmoid) + reweighting**  
   The sigmoid outputs weights in **[0, 1]**, applied channel-wise:
   - weights reshaped to **[B, C, 1, 1]**
   - output: **x ⊙ weights**

### Why it matters in TEKO docking
Different channels encode different cues (e.g., edges, contrast patterns, connector geometry).  
Channel attention lets the network amplify the channels that best correlate with docking success and suppress noisy or irrelevant features.


In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        b, c, h, w = x.shape
        y = x.view(b, c, -1).mean(-1)
        return x * self.fc(y).view(b, c, 1, 1)

## 5. Vision Encoder with Dual Attention + Auxiliary Yaw Head

This encoder maps an image observation into a compact **feature vector** used by the policy/value networks, and also produces an **auxiliary yaw prediction** to enforce orientation-aware representations.

### Input
- Observation: **128 × 128 × 4** (e.g., RGB + Depth)  
- Tensor layout in PyTorch: **[B, 4, 128, 128]**

### Convolutional backbone (with GroupNorm)
| Layer | Kernel / Stride / Pad | Output shape (C×H×W) |
|---|---|---|
| Conv1 + GN + ReLU | 8×8 / 4 / 2 | 32 × 32 × 32 |
| Conv2 + GN + ReLU | 4×4 / 2 / 1 | 64 × 16 × 16 |
| Conv3 + GN + ReLU | 3×3 / 1 / 1 | 64 × 16 × 16 |

**Why GroupNorm (instead of BatchNorm):** RL batches are often non-i.i.d. and can have small/variable effective batch sizes. GroupNorm is typically more stable in this regime.

### Attention (applied after Conv3)
- **Channel Attention:** learns *which feature channels* are important  
- **Spatial Attention:** learns *where in the feature map* to focus  
Applied sequentially on the **64 × 16 × 16** feature map.

### Feature projection
- Flatten: **64 × 16 × 16 = 16384**
- Linear projection: **16384 → 256** (feature vector)

**Initialization:** orthogonal initialization is used for stability in RL.

### Auxiliary yaw head
A small MLP predicts yaw error from the 256D features:
- Output uses `tanh` → **[-1, 1]**
- Scaled by `π` → **[-π, π]** radians

**Why:** docking requires strong orientation awareness; the yaw auxiliary task encourages the encoder to learn features predictive of relative heading, improving policy learning under large yaw offsets.


In [ ]:
class VisionEncoderAttentionYaw(nn.Module):
    def __init__(self, in_channels=4, feature_dim=256):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, 8, stride=4, padding=2)
        self.conv2 = nn.Conv2d(32, 64, 4, stride=2, padding=1)
        self.conv3 = nn.Conv2d(64, 64, 3, stride=1, padding=1)
        self.channel_attn = ChannelAttention(64)
        self.spatial_attn = SpatialAttention(64)
        self.gn1 = nn.GroupNorm(8, 32)
        self.gn2 = nn.GroupNorm(8, 64)
        self.gn3 = nn.GroupNorm(8, 64)
        
        self._init_weights()
        
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, 128, 128)
            flat_size = self._forward_conv(dummy).shape[1]
        
        self.fc = nn.Linear(flat_size, feature_dim)
        nn.init.orthogonal_(self.fc.weight, gain=1.0)
        nn.init.zeros_(self.fc.bias)
        
        self.yaw_head = nn.Sequential(
            nn.Linear(feature_dim, 64), nn.ReLU(True),
            nn.Linear(64, 32), nn.ReLU(True),
            nn.Linear(32, 1), nn.Tanh()
        )
        self.feature_dim = feature_dim
    
    def _init_weights(self):
        for m in [self.conv1, self.conv2, self.conv3]:
            nn.init.orthogonal_(m.weight, gain=nn.init.calculate_gain('relu'))
            nn.init.zeros_(m.bias)
    
    def _forward_conv(self, x):
        x = F.relu(self.gn1(self.conv1(x)))
        x = F.relu(self.gn2(self.conv2(x)))
        x = F.relu(self.gn3(self.conv3(x)))
        x = self.channel_attn(x)
        x = self.spatial_attn(x)
        return x.flatten(1)
    
    def forward(self, x):
        return F.relu(self.fc(self._forward_conv(x)))
    
    def predict_yaw(self, features):
        return self.yaw_head(features) * math.pi

## 6. Full Policy Network (Actor–Critic)

This module implements an **actor–critic** policy with:
- **Multi-modal fusion:** RGB-D vision + IMU
- **Asymmetric critic:** critic receives additional privileged state (training-only)
- **Auxiliary yaw head:** yaw prediction from vision features (regularizes representation)

---

### 6.1 Encoders and feature fusion
- **Vision encoder:** RGB-D → **256D** (`vis_dim`)
- **IMU encoder:** 6D IMU (lin + ang velocity) → **64D**
- **Fused feature:** concatenation → **256 + 64 = 320D**

---

### 6.2 Actor (policy)
The actor outputs a Gaussian distribution over actions:
- Mean: `actor_head(fused)`
- Std: global learnable parameter `log_std` (clamped for stability)

Actions are **squashed with `tanh`** to enforce bounds **[-1, 1]**.

**Log-prob correction:** because of the `tanh` squashing, the log probability includes a Jacobian correction term.

---

### 6.3 Critic (value function) — asymmetric training
The critic sees:
- fused features (**320D**) + privileged state (**7D**) → **327D**

Privileged state contains ground-truth quantities (e.g., relative pose/yaw) that are **not available to the actor**.  
This typically improves value estimation and stabilizes PPO training.

---

### 6.4 Auxiliary yaw prediction
The yaw head predicts yaw error in **[-π, π]** from the vision features.  
This encourages the encoder to encode orientation cues important for docking from arbitrary angles.

---

### 6.5 PPO evaluation step
During PPO updates, stored actions are re-evaluated by:
1. mapping actions back to pre-squash space via `atanh`
2. computing log-probability under the Gaussian
3. computing entropy for exploration regularization


In [ ]:
class VisionIMUAttentionYawPolicy(nn.Module):
    LOG_STD_MIN, LOG_STD_MAX = -2.0, 0.5
    
    def __init__(self, vis_dim=256, imu_dim=6, hidden=256, action_dim=2):
        super().__init__()
        self.vision_encoder = VisionEncoderAttentionYaw(in_channels=4, feature_dim=vis_dim)
        self.imu_encoder = nn.Sequential(
            nn.Linear(imu_dim, 64), nn.ReLU(True),
            nn.Linear(64, 64), nn.ReLU(True),
        )
        fused_dim = vis_dim + 64
        self.actor_head = nn.Sequential(
            nn.Linear(fused_dim, hidden), nn.ReLU(True),
            nn.Linear(hidden, action_dim),
        )
        self.log_std = nn.Parameter(torch.full((action_dim,), -0.5))
        priv_dim = 7
        self.critic_head = nn.Sequential(
            nn.Linear(fused_dim + priv_dim, hidden), nn.ReLU(True),
            nn.Linear(hidden, hidden // 2), nn.ReLU(True),
            nn.Linear(hidden // 2, 1),
        )
    
    def _std(self):
        return torch.exp(torch.clamp(self.log_std, self.LOG_STD_MIN, self.LOG_STD_MAX))
    
    def forward_features(self, rgb, imu):
        vis_feat = self.vision_encoder(rgb)
        imu_feat = self.imu_encoder(imu)
        return torch.cat([vis_feat, imu_feat], dim=-1), vis_feat
    
    def act(self, rgb, imu, privileged=None, deterministic=False):
        fused, vis_feat = self.forward_features(rgb, imu)
        mean = self.actor_head(fused)
        std = self._std().unsqueeze(0).expand_as(mean)
        dist = torch.distributions.Normal(mean, std)
        u = dist.mean if deterministic else dist.rsample()
        action = torch.tanh(u)
        log_prob = dist.log_prob(u).sum(-1) - torch.log(1 - action.pow(2) + 1e-6).sum(-1)
        
        if privileged is not None:
            critic_in = torch.cat([fused, privileged], dim=-1)
        else:
            critic_in = torch.cat([fused, torch.zeros(fused.shape[0], 7, device=fused.device)], dim=-1)
        value = self.critic_head(critic_in).squeeze(-1)
        yaw_pred = self.vision_encoder.predict_yaw(vis_feat)
        return action, log_prob, value, yaw_pred
    
    def evaluate(self, rgb, imu, actions, privileged=None):
        fused, vis_feat = self.forward_features(rgb, imu)
        mean = self.actor_head(fused)
        std = self._std().unsqueeze(0).expand_as(mean)
        dist = torch.distributions.Normal(mean, std)
        u = torch.clamp(actions, -0.999, 0.999)
        u = 0.5 * (torch.log1p(u) - torch.log1p(-u))
        log_prob = dist.log_prob(u).sum(-1) - torch.log(1 - actions.pow(2) + 1e-6).sum(-1)
        entropy = dist.entropy().sum(-1)
        
        if privileged is not None:
            critic_in = torch.cat([fused, privileged], dim=-1)
        else:
            critic_in = torch.cat([fused, torch.zeros(fused.shape[0], 7, device=fused.device)], dim=-1)
        value = self.critic_head(critic_in).squeeze(-1)
        yaw_pred = self.vision_encoder.predict_yaw(vis_feat)
        return log_prob, value, entropy, yaw_pred

## 7. Generalized Advantage Estimation (GAE-λ)

This function computes advantages using **GAE-λ**, which reduces variance compared to pure Monte Carlo returns while keeping bias controlled.

### Temporal-difference error (TD residual)
\[
\delta_t = r_t + \gamma \, V(s_{t+1}) \, (1-d_t) - V(s_t)
\]
where \(d_t \in \{0,1\}\) indicates episode termination.

### GAE advantage
\[
A_t = \delta_t + (\gamma \lambda)\delta_{t+1} + (\gamma \lambda)^2\delta_{t+2} + \dots
\]

### Interpretation of λ
- **λ = 0** → pure TD (higher bias, lower variance)  
- **λ = 1** → Monte Carlo (lower bias, higher variance)  
- **λ = 0.94** → practical bias–variance tradeoff (used here)

### Outputs
- **advantages**: \(A_t\)
- **returns**: \(R_t = A_t + V(s_t)\)


In [ ]:
def compute_gae(rewards, values, dones, gamma, lam, last_value):
    T, N = rewards.shape
    advantages = torch.zeros_like(rewards)
    last_gae = torch.zeros(N, device=rewards.device)
    for t in reversed(range(T)):
        next_val = last_value if t == T - 1 else values[t + 1]
        delta = rewards[t] + gamma * next_val * (1 - dones[t]) - values[t]
        last_gae = delta + gamma * lam * (1 - dones[t]) * last_gae
        advantages[t] = last_gae
    return advantages, advantages + values

## 8. PPO Update with Auxiliary Yaw Loss

This step performs the PPO optimization over a rollout buffer and adds an **auxiliary yaw regression loss** to regularize the vision encoder.

---

### 8.1 Buffer flattening and advantage normalization
Rollout tensors are stored as **[T, N, ...]** (time × parallel envs).  
For minibatch SGD, everything is flattened to **[T·N, ...]**.

Advantages are normalized:
\[
\hat{A} = \frac{A - \mu(A)}{\sigma(A) + \epsilon}
\]
This improves optimization stability.

---

### 8.2 PPO clipped objective
Let:
\[
r_t(\theta)=\frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}
\]

The clipped surrogate:
\[
L^{\text{CLIP}} = \mathbb{E}\left[\min\left(r_t \hat{A}_t,\;\text{clip}(r_t, 1-\epsilon, 1+\epsilon)\hat{A}_t\right)\right]
\]

Policy loss (to minimize):
\[
\mathcal{L}_{\pi} = -L^{\text{CLIP}}
\]

---

### 8.3 Value loss and entropy bonus
Value regression:
\[
\mathcal{L}_V = \frac{1}{2}\,\text{MSE}(V_\phi(s), R)
\]

Entropy bonus encourages exploration:
\[
\mathcal{L}_H = -\beta \, \mathbb{E}[H(\pi_\theta)]
\]

---

### 8.4 Auxiliary yaw loss
Yaw prediction is supervised with MSE:
\[
\mathcal{L}_{yaw} = \text{MSE}(\hat{y}, y)
\]
where yaw is expressed in radians and predicted in **[-π, π]**.

---

### 8.5 Total loss
\[
\mathcal{L} = \mathcal{L}_{\pi} + c_V \mathcal{L}_V - c_H \mathbb{E}[H] + c_{yaw}\mathcal{L}_{yaw}
\]
with coefficients from `CONFIG`:
- `value_coef` = \(c_V\)
- `entropy_coef` = \(c_H\)
- `aux_yaw_coef` = \(c_{yaw}\)

Gradient clipping (`max_grad_norm`) is applied to prevent instability.


In [ ]:
def ppo_update_with_yaw(policy, optimizer, rgb, imu, actions, old_logp, advantages, returns, yaw_targets, privileged, cfg):
    device = next(policy.parameters()).device
    T, N = rgb.shape[:2]
    total = T * N
    
    rgb_flat = rgb.view(total, *rgb.shape[2:])
    imu_flat = imu.view(total, -1)
    actions_flat = actions.view(total, -1)
    old_logp_flat = old_logp.view(total)
    adv_flat = (advantages.view(total) - advantages.mean()) / (advantages.std() + 1e-8)
    ret_flat = returns.view(total)
    yaw_flat = yaw_targets.view(total, 1)
    priv_flat = privileged.view(total, -1) if privileged is not None else None
    
    metrics = {"policy_loss": 0, "value_loss": 0, "entropy": 0, "yaw_loss": 0, "grad_norm": 0}
    n_updates = 0
    
    for _ in range(cfg["epochs"]):
        idx = torch.randperm(total, device=device)
        for start in range(0, total, cfg["batch_size"]):
            mb = idx[start:start + cfg["batch_size"]]
            priv_mb = priv_flat[mb] if priv_flat is not None else None
            logp, val, ent, yaw_pred = policy.evaluate(rgb_flat[mb], imu_flat[mb], actions_flat[mb], priv_mb)
            
            ratio = torch.exp(logp - old_logp_flat[mb])
            surr1 = ratio * adv_flat[mb]
            surr2 = torch.clamp(ratio, 1 - cfg["clip_ratio"], 1 + cfg["clip_ratio"]) * adv_flat[mb]
            p_loss = -torch.min(surr1, surr2).mean()
            v_loss = 0.5 * F.mse_loss(val, ret_flat[mb])
            yaw_loss = F.mse_loss(yaw_pred, yaw_flat[mb])
            
            loss = p_loss + cfg["value_coef"] * v_loss - cfg["entropy_coef"] * ent.mean() + cfg["aux_yaw_coef"] * yaw_loss
            
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            grad_norm = nn.utils.clip_grad_norm_(policy.parameters(), cfg["max_grad_norm"]).item()
            optimizer.step()
            
            metrics["policy_loss"] += p_loss.item()
            metrics["value_loss"] += v_loss.item()
            metrics["entropy"] += ent.mean().item()
            metrics["yaw_loss"] += yaw_loss.item()
            metrics["grad_norm"] += grad_norm
            n_updates += 1
    
    return {k: v / max(n_updates, 1) for k, v in metrics.items()}

## 9–18. Training Pipeline (Initialization → Rollouts → PPO → Curriculum → Logging)

This section initializes reproducibility, launches Isaac Lab in headless mode, sets up logging, allocates rollout buffers on GPU, and runs the PPO training loop with:
- **GAE-λ advantages**
- **PPO clipped objective**
- **Asymmetric critic** (privileged state for critic only, training-time)
- **Auxiliary yaw regression loss** (regularizes the vision encoder)
- **Curriculum advancement** based on success-rate threshold

---

### 9. Training Initialization
- Fix seeds for reproducibility (PyTorch / NumPy / Python)
- `cudnn.benchmark=True` to select faster convolution algorithms when input sizes are fixed
- Force GPU device (`cuda:0`)

### 10. Logging Setup
- Timestamped TensorBoard directory + CSV file for thesis figures
- Log scalars: SSR, reward, losses, entropy, grad norm, stage, elapsed hours

### 11. Rollout Buffers
Pre-allocated GPU buffers with shape **[T, N, ...]**, where:
- `T = rollout_len` (e.g., 128)
- `N = num_envs` (e.g., 120)
Total transitions per rollout: **T×N = 15,360**

### 12–13. Main Loop + Rollout Collection
For each rollout step:
- Read observations (`rgb`, `imu`, optional `privileged`)
- Sample action from policy (no gradients)
- Store transition in buffers
- Step environment
- Track episode rewards + rolling success rate (SSR)

### 14. GAE + PPO Update
- Bootstrap last value estimate
- Compute `advantages, returns = compute_gae(...)`
- Run `ppo_update_with_yaw(...)`

### 15. Curriculum Advancement
Advance stage if:
- at least 100 completed episodes in the rolling window
- `SSR ≥ advance_threshold` (e.g., 0.75)
- `min_steps_before_advance` satisfied (e.g., 200k)
- not yet at `max_stage`

### 16. Logging & Checkpointing
- Every `log_interval`: print + TensorBoard + CSV
- Every `save_interval`: save checkpoint (policy + optimizer + config + stage)

### 17. Early Stop & Cleanup
- Early success stop (e.g., stage 41 with SSR ≥ 0.70)
- Always save a final checkpoint and close writer/env/sim in `finally`

---

### Summary Diagram (high level)



In [ ]:
def train(args):
    # Fixed seed for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    random.seed(42)
    torch.backends.cudnn.benchmark = True
    device = torch.device("cuda:0")
    
    app = AppLauncher(args)
    sim = app.app
    
    sys.path.insert(0, "/workspace/teko/source/teko")
    from teko.tasks.direct.teko.teko_env_tiled_imu import TekoEnvTiledIMU
    from teko.tasks.direct.teko.teko_env_cfg import TekoEnvCfg
    
    cfg = TekoEnvCfg()
    cfg.scene.num_envs = CONFIG["num_envs"]
    cfg.enable_curriculum = True
    cfg.asymmetric_critic = True
    
    env = TekoEnvTiledIMU(cfg=cfg)
    
    policy = VisionIMUAttentionYawPolicy().to(device)
    optimizer = torch.optim.Adam(policy.parameters(), lr=CONFIG["learning_rate"])
    
    # TensorBoard setup
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_dir = f"/home/schux00/tensorboard/vision_optimal_{timestamp}"
    csv_path = f"/home/schux00/logs/vision_optimal_{timestamp}.csv"
    
    writer = None
    if HAS_TENSORBOARD:
        os.makedirs(log_dir, exist_ok=True)
        writer = SummaryWriter(log_dir)
        print(f"[TB] Logging to {log_dir}")
    
    # CSV logging
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    csv_file = open(csv_path, 'w', newline='')
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['step', 'stage', 'ssr', 'reward', 'entropy', 'yaw_loss', 'policy_loss', 'value_loss', 'hours'])
    
    num_envs = CONFIG["num_envs"]
    rollout_len = CONFIG["rollout_len"]
    
    # Buffers
    rgb_buf = torch.zeros((rollout_len, num_envs, 4, 128, 128), device=device)
    imu_buf = torch.zeros((rollout_len, num_envs, 6), device=device)
    actions_buf = torch.zeros((rollout_len, num_envs, 2), device=device)
    rewards_buf = torch.zeros((rollout_len, num_envs), device=device)
    values_buf = torch.zeros((rollout_len, num_envs), device=device)
    logprobs_buf = torch.zeros((rollout_len, num_envs), device=device)
    dones_buf = torch.zeros((rollout_len, num_envs), device=device)
    yaw_targets_buf = torch.zeros((rollout_len, num_envs, 1), device=device)
    priv_buf = torch.zeros((rollout_len, num_envs, 7), device=device)
    
    ep_rewards = deque(maxlen=300)
    stage_successes = deque(maxlen=300)
    cur_reward = torch.zeros(num_envs, device=device)
    
    current_stage = 0
    max_stage_reached = 0
    last_advance_step = 0
    
    obs_dict, _ = env.reset()
    step = 0
    t0 = time.time()
    next_log = CONFIG["log_interval"]
    next_save = CONFIG["save_interval"]
    
    print("=" * 70)
    print("TEKO Vision + Attention + YawAux - OPTIMAL (Final)")
    print("=" * 70)
    print(f"Host: {socket.gethostname()}")
    print(f"Envs: {num_envs} | Max Steps: {CONFIG['max_steps']:,}")
    print(f"LR: {CONFIG['learning_rate']} | Entropy: {CONFIG['entropy_coef']}")
    print(f"GAE Lambda: {CONFIG['gae_lambda']} | Batch: {CONFIG['batch_size']}")
    print(f"YawAux Coef: {CONFIG['aux_yaw_coef']}")
    print(f"TensorBoard: {log_dir}")
    print(f"CSV: {csv_path}")
    print("=" * 70)
    
    has_privileged = "privileged" in obs_dict and obs_dict["privileged"] is not None
    
    try:
        while step < CONFIG["max_steps"]:
            elapsed_h = (time.time() - t0) / 3600
            if elapsed_h > CONFIG["max_hours"]:
                print(f"[TIME] Reached {CONFIG['max_hours']}h limit")
                break
            
            for t in range(rollout_len):
                rgb = obs_dict["rgb"].to(device)
                imu = obs_dict["imu"].to(device)
                priv = obs_dict.get("privileged")
                if priv is not None:
                    priv = priv.to(device)
                    yaw_target = priv[:, 3:4]
                else:
                    yaw_target = torch.zeros(num_envs, 1, device=device)
                
                with torch.no_grad():
                    action, logp, value, _ = policy.act(rgb, imu, priv)
                
                rgb_buf[t] = rgb
                imu_buf[t] = imu
                actions_buf[t] = action
                logprobs_buf[t] = logp
                values_buf[t] = value
                yaw_targets_buf[t] = yaw_target
                if priv is not None:
                    priv_buf[t] = priv
                
                obs_dict, reward, term, trunc, info = env.step(action)
                done = term | trunc
                
                rewards_buf[t] = reward
                dones_buf[t] = done.float()
                cur_reward += reward
                
                if done.any():
                    done_idx = done.nonzero(as_tuple=False).squeeze(-1)
                    if hasattr(env, "_last_success"):
                        succ = env._last_success.float()
                    else:
                        _, _, sxy, _ = env.get_sphere_distances_from_physics()
                        succ = (sxy < 0.03).float()
                    ep_rewards.extend(cur_reward[done_idx].cpu().tolist())
                    stage_successes.extend(succ[done_idx].cpu().tolist())
                    cur_reward[done_idx] = 0
                
                step += num_envs
            
            with torch.no_grad():
                last_rgb = obs_dict["rgb"].to(device)
                last_imu = obs_dict["imu"].to(device)
                last_priv = obs_dict.get("privileged")
                if last_priv is not None:
                    last_priv = last_priv.to(device)
                _, _, last_value, _ = policy.act(last_rgb, last_imu, last_priv)
            
            advantages, returns = compute_gae(
                rewards_buf, values_buf, dones_buf,
                CONFIG["gamma"], CONFIG["gae_lambda"], last_value
            )
            
            metrics = ppo_update_with_yaw(
                policy, optimizer, rgb_buf, imu_buf, actions_buf, logprobs_buf,
                advantages, returns, yaw_targets_buf,
                priv_buf if has_privileged else None, CONFIG
            )
            
            ssr = float(np.mean(stage_successes)) if stage_successes else 0.0
            mean_r = float(np.mean(ep_rewards)) if ep_rewards else 0.0
            
            # Curriculum advancement
            if (len(stage_successes) >= 100 and
                ssr >= CONFIG["advance_threshold"] and
                step - last_advance_step >= CONFIG["min_steps_before_advance"] and
                current_stage < CONFIG["max_stage"]):
                
                print(f"[ADVANCE] Stage {current_stage} -> {current_stage + 1} (SSR={ssr:.1%})")
                current_stage += 1
                max_stage_reached = max(max_stage_reached, current_stage)
                env.set_curriculum_level(current_stage)
                stage_successes.clear()
                last_advance_step = step
                
                if writer:
                    writer.add_scalar("curriculum/stage", current_stage, step)
            
            # Logging
            if step >= next_log:
                print(f"[{step:,}] S{current_stage:02d} | SSR: {ssr:.1%} | R: {mean_r:.1f} | "
                      f"YawL: {metrics['yaw_loss']:.3f} | Ent: {metrics['entropy']:.3f} | "
                      f"MaxS: {max_stage_reached} | {elapsed_h:.1f}h")
                
                # TensorBoard
                if writer:
                    writer.add_scalar("train/ssr", ssr, step)
                    writer.add_scalar("train/reward", mean_r, step)
                    writer.add_scalar("train/entropy", metrics["entropy"], step)
                    writer.add_scalar("train/yaw_loss", metrics["yaw_loss"], step)
                    writer.add_scalar("train/policy_loss", metrics["policy_loss"], step)
                    writer.add_scalar("train/value_loss", metrics["value_loss"], step)
                    writer.add_scalar("train/grad_norm", metrics["grad_norm"], step)
                    writer.add_scalar("curriculum/stage", current_stage, step)
                    writer.add_scalar("curriculum/max_stage", max_stage_reached, step)
                
                # CSV
                csv_writer.writerow([step, current_stage, f"{ssr:.4f}", f"{mean_r:.2f}",
                                    f"{metrics['entropy']:.4f}", f"{metrics['yaw_loss']:.4f}",
                                    f"{metrics['policy_loss']:.4f}", f"{metrics['value_loss']:.4f}",
                                    f"{elapsed_h:.2f}"])
                csv_file.flush()
                
                next_log += CONFIG["log_interval"]
            
            # Save checkpoint
            if step >= next_save:
                ckpt_path = f"/home/schux00/checkpoints/vision_optimal_S{current_stage}_{step//1000}k.pt"
                os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
                torch.save({
                    "step": step,
                    "stage": current_stage,
                    "max_stage": max_stage_reached,
                    "policy": policy.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "config": CONFIG,
                }, ckpt_path)
                print(f"[SAVE] {ckpt_path}")
                next_save += CONFIG["save_interval"]
            
            # Early success
            if current_stage >= CONFIG["max_stage"] and ssr >= 0.70:
                print("=" * 70)
                print(f"[SUCCESS] Reached Stage {CONFIG['max_stage']} with SSR={ssr:.1%}!")
                print("=" * 70)
                break
    
    except KeyboardInterrupt:
        print("\n[INTERRUPTED]")
    except Exception as e:
        print(f"[ERROR] {e}")
        import traceback
        traceback.print_exc()
    
    finally:
        # Final save
        final_path = f"/home/schux00/checkpoints/vision_optimal_FINAL_S{max_stage_reached}.pt"
        torch.save({
            "step": step,
            "stage": current_stage,
            "max_stage": max_stage_reached,
            "policy": policy.state_dict(),
            "config": CONFIG,
        }, final_path)
        print(f"[FINAL] Saved to {final_path}")
        print(f"[DONE] MaxStage={max_stage_reached}, Steps={step:,}, Time={(time.time()-t0)/3600:.1f}h")
        
        if writer:
            writer.close()
        csv_file.close()
        env.close()
        sim.close()

## Notes before running `main()`

### Execution mode
This script is designed to run **headless** (no GUI) with **cameras enabled**, which is required for RGB-D observations in Isaac Lab.

### Privileged state (asymmetric critic)
During training, the critic can receive a *privileged* vector (dim = **7**) that is **not available to the actor**.
This is used only to stabilize value estimation and speed up learning.

- `privileged[:, 3]` is used as the **yaw error target** for the auxiliary yaw loss.
- If `privileged` is not provided by the environment, the code falls back to zeros.

### Reproducibility
Seeds are fixed (`torch`, `numpy`, `random`) and `cudnn.benchmark=True` is enabled for faster conv kernels with fixed input sizes.


In [ ]:
def main():
    parser = argparse.ArgumentParser()
    AppLauncher.add_app_launcher_args(parser)
    args = parser.parse_args()
    args.headless = True
    args.enable_cameras = True
    train(args)


if __name__ == "__main__":
    main()

<pre style="white-space: pre; overflow-x: auto; font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace;">
┌───────────────────────────────────────────────────────────────────────┐
│                               OBSERVATIONS                            │
│   RGB-D (4×128×128)                 IMU (6D)         Privileged (7D)  │
└───────────────┬───────────────────────┬───────────────────────┬───────┘
                │                       │                       │
                ▼                       ▼                       │
        ┌────────────────┐       ┌───────────────┐              │
        │ Vision Encoder │       │  IMU Encoder  │              │
        │ Conv+GN        │       │  MLP (6→64)   │              │
        │ + ChAttn       │       └───────┬───────┘              │
        │ + SpAttn       │               │                      │
        │ + FC (→256D)   │               │                      │
        └───────┬────────┘               │                      │
                │                        │                      │
                ├──────────────┐         │                      │
                │              ▼         ▼                      │
                │       ┌───────────────────────────┐           │
                │       │   Fuse (concat) = 320D    │◄──────────┘
                │       │  (256 vision + 64 IMU)    │
                │       └───────────┬───────────────┘
                │                   │
                │                   ├─────────────────────────────┐
                │                   │                             │
                ▼                   ▼                             ▼
      ┌─────────────────┐   ┌──────────────────────┐     ┌──────────────────────┐
      │ Aux Yaw Head    │   │        ACTOR         │     │        CRITIC        │
      │ (from 256D)     │   │ MLP → mean           │     │ (asymmetric)         │
      │ tanh → [-1,1]   │   │ + learnable log_std  │     │ input: 320D + 7D     │
      │ × π → [-π, π]   │   │ Normal → tanh(action)│     │ MLP → V(s)           │
      └───────┬─────────┘   └──────────┬───────────┘     └───────────┬──────────┘
              │                        │                             │
              ▼                        ▼                             ▼
      yaw_pred (rad)             action [v, ω]                  value V(s)
              │                        │                             │
              └───────────────┬────────┴───────────────┬─────────────┘
                              ▼                        ▼
                     ┌───────────────────────────────────────────┐
                     │                 PPO UPDATE                │
                     │  GAE-λ → advantages/returns               │
                     │  Lπ (clipped) + cV LV - cH H + cYaw Lyaw  │
                     │  Adam + grad clipping                     │
                     └───────────────────────────────────────────┘

</pre>